# Préparation des données DPO — UltraMedical-Preference

Construction du jeu de paires préférentielles (`prompt` / `chosen` / `rejected`) pour l'alignement DPO, à partir de [TsinghuaC3I/UltraMedical-Preference](https://huggingface.co/datasets/TsinghuaC3I/UltraMedical-Preference) (anglais, licence MIT).

- **Origine des paires** : réponses générées par plusieurs LLM (GPT-4, Llama-3, Mixtral, Qwen…), notées et classées par GPT-4. Le split `test` (777 paires) a été **revu par des experts biomédicaux** (Medical RewardBench).
- **Qualité plutôt que quantité** : les paires des sources QCM ne sont gardées que si le **corrigé** (`golden_answer`) confirme que la `chosen` est juste et la `rejected` fausse. Les autres sources, sans corrigé, ne gardent que les paires où GPT-4 exprime une préférence nette.
- **Étapes** : analyse → filtre sur `label_type` → vérification par le corrigé → nettoyage → filtre sur la longueur en tokens → échantillonnage → JSONL → anonymisation.
- **Sortie** : `data/anonymized/dpo_ultramedical_{train,validation,test}.anonymized.jsonl`, au format conversationnel attendu par `trl.DPOTrainer`.

# Import

In [1]:
import json
import re

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer

from projet_14.rgpd import run

SEED = 42
MODEL_NAME = "Qwen/Qwen3-1.7B-Base"  # pour compter les tokens avec le tokenizer du modèle entraîné

# Types de paires conservés pour train / validation (voir l'analyse plus bas)
LABEL_TYPES = ["hard", "length"]

# Sources QCM : la bonne réponse (golden_answer) est une lettre, vérifiable dans les réponses
MCQ_SOURCES = ["MedQA", "MedMCQA", "MedQA-Evol", "PubMedQA", "TextBookQA"]
# Sources sans corrigé : seuils sur les notes GPT-4 (de 1 à 5)
MIN_CHOSEN_SCORE = 4.5
MIN_SCORE_GAP = 2

# Longueurs maximales en tokens : à reprendre dans DPOConfig (max_prompt_length, max_length)
MAX_PROMPT_TOKENS = 512
MAX_LENGTH = 1024  # prompt + réponse

# Taille maximale des jeux après filtrage (le test est gardé entier)
N_TRAIN = 2000
N_VALIDATION = 300

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Environ 1 Go à télécharger (fichier train.json)
ds = load_dataset("TsinghuaC3I/UltraMedical-Preference")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 109353
    })
    validation: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 2232
    })
    test: Dataset({
        features: ['prompt_id', 'label_type', 'prompt', 'chosen', 'rejected', 'metadata', 'feedback'],
        num_rows: 777
    })
})


In [3]:
def flatten(df: pd.DataFrame) -> pd.DataFrame:
    """Extrait en colonnes la source, les réponses et les notes GPT-4 de chaque paire."""
    meta = pd.json_normalize(df["metadata"].tolist())
    return pd.DataFrame(
        {
            "prompt_id": df["prompt_id"],
            "source": df["prompt_id"].str.split(",").str[0],  # "MedQA,1234" -> "MedQA"
            "label_type": df["label_type"],
            "prompt": df["prompt"],
            # chosen / rejected contiennent toute la conversation : [user, assistant]
            "chosen": df["chosen"].map(lambda messages: messages[-1]["content"]),
            "rejected": df["rejected"].map(lambda messages: messages[-1]["content"]),
            "chosen_model": meta["chosen.model"],
            "rejected_model": meta["rejected.model"],
            "chosen_score": meta["chosen.score"],
            "rejected_score": meta["rejected.score"],
            "golden_answer": meta["golden_answer"],  # lettre pour les QCM, texte libre ou vide sinon
        }
    )


splits = {name: flatten(ds[name].to_pandas()) for name in ds}
df_train = splits["train"]

# Analyse dataset

In [4]:
df_train.head()

,prompt_id,source,label_type,prompt,chosen,rejected,chosen_model,rejected_model,chosen_score,rejected_score,golden_answer
0,"WikiInstruct,8304",WikiInstruct,length,Investigate the intricacies of immunometabolis...,Immunometabolism is an emerging field of stud...,Immunometabolism is an emerging field that exp...,Mixtral-8x7B-Instruct,Llama-3-8B-UltraMedical,5.0,4.0,NaN
1,"WikiInstruct,8846",WikiInstruct,length,Examine the differential immunotoxic effects e...,Title: Differential Immunotoxic Effects of Mi...,The examination of differential immunotoxic ef...,Mixtral-8x7B-Instruct,Llama-3-8B-UltraMedical,5.0,4.0,NaN
2,"MedMCQA,11404",MedMCQA,length,A lady comes with melanotic pigmentation of li...,\n\nLet's think step by step.\n\nThe patient h...,Let's break down the symptoms:\n\n* Melanotic ...,Meta-Llama-3-70B-Instruct,Meta-Llama-3-8B-Instruct,5.0,4.0,A
3,"ChatDoctor,33567",ChatDoctor,easy,my daughter was diagnosed with a liver cyst th...,"I'm not a medical professional, but I can prov...","I'm not a medical professional, but it sounds ...",gpt-4-1106-preview,gpt-3.5-turbo-1106,3.0,1.0,"Hi, Thank you for your query. Amoebic liver ab..."
4,"WikiInstruct,14624",WikiInstruct,length,Investigate the physiological mechanisms under...,\n\nThe blood-brain barrier (BBB) is a complex...,The blood-brain barrier (BBB) is a complex phy...,Meta-Llama-3-70B-Instruct,Meta-Llama-3-8B-Instruct,4.0,3.0,NaN


In [5]:
# Nombres de lignes dans les différentes parties du dataset
for name, df in splits.items():
    print(f"{name}: {len(df)} rows")

train: 109353 rows
validation: 2232 rows
test: 777 rows


In [6]:
# Types de paires par split ("human" : paires revues par des experts, uniquement dans le test)
pd.concat({name: df["label_type"].value_counts() for name, df in splits.items()}, axis=1).fillna(0).astype(int)

,train,validation,test
label_type,,,
hard,45765,925,196
length,41130,859,180
easy,22458,448,238
human,0,0,163


In [7]:
# Sources des questions, par type de paire
pd.crosstab(df_train["source"], df_train["label_type"], margins=True)

label_type,easy,hard,length,All
source,,,,
ChatDoctor,1883,5273,4153,11309
MedInstruct-52k,570,1363,908,2841
MedMCQA,4186,6540,6828,17554
MedQA,2730,3784,4289,10803
MedQA-Evol,2273,3692,3274,9239
MedQuad,1323,2414,2118,5855
Medical-Instruct-120k,789,2316,1847,4952
PubMedQA,538,1718,1037,3293
TextBookQA,3670,6301,6043,16014


In [8]:
# Caractéristiques de chaque type de paire (train)
stats = df_train.assign(
    score_gap=df_train["chosen_score"] - df_train["rejected_score"],
    chosen_longer=df_train["chosen"].str.len() > df_train["rejected"].str.len(),
)
stats.groupby("label_type").agg(
    pairs=("prompt", "size"),
    chosen_score=("chosen_score", "mean"),
    rejected_score=("rejected_score", "mean"),
    score_gap=("score_gap", "mean"),
    gap_le_0=("score_gap", lambda s: (s <= 0).mean()),
    chosen_longer=("chosen_longer", "mean"),
).round(2)

,pairs,chosen_score,rejected_score,score_gap,gap_le_0,chosen_longer
label_type,,,,,,
easy,22458,4.59,3.31,1.28,0.00,0.90
hard,45765,4.74,3.76,0.98,0.01,0.78
length,41130,4.39,3.03,1.36,0.15,0.38


Lecture du tableau :

- **easy** : grand écart de note, mais la réponse `chosen` est la plus longue dans ~90 % des cas. Le modèle apprendrait surtout que « plus long = meilleur », à l'opposé de réponses de triage courtes et claires.
- **hard** : deux bonnes réponses, avec un écart d'un point environ. Distinctions fines sur le fond.
- **length** : paires construites pour corriger le biais de longueur (la `chosen` est souvent la plus courte). Environ 15 % ont un écart de note ≤ 0 : ce sont des paires bruitées, à écarter.

In [9]:
# Doublons : une même question peut apparaître dans plusieurs paires (easy, hard, length)
print(f"Questions en double dans train : {df_train['prompt'].duplicated().sum()}")

# Fuite entre splits : questions du train présentes aussi en validation ou en test
for other in ["validation", "test"]:
    print(f"Questions du train présentes en {other} : {df_train['prompt'].isin(splits[other]['prompt']).sum()}")

Questions en double dans train : 32829
Questions du train présentes en validation : 1664
Questions du train présentes en test : 43


In [10]:
ANSWER = re.compile(r"answer is:?\s*\(?\**([A-J])\b", re.IGNORECASE)  # « So, the answer is C. »


def answer_letter(text: str) -> str | None:
    """Lettre donnée comme réponse finale, ou None si absente."""
    found = ANSWER.findall(text)
    return found[-1].upper() if found else None


def gold_check(df: pd.DataFrame) -> pd.Series:
    """Compare les réponses des QCM au corrigé : catégorie de chaque paire (NaN hors QCM)."""
    chosen, rejected = df["chosen"].map(answer_letter), df["rejected"].map(answer_letter)
    check = pd.Series("chosen fausse", index=df.index)
    check[chosen.isna() | rejected.isna()] = "lettre introuvable"
    check[(chosen == df["golden_answer"]) & (rejected == df["golden_answer"])] = "les deux justes"
    check[(chosen == df["golden_answer"]) & rejected.notna() & (rejected != df["golden_answer"])] = "vérifiée"
    return check.where(df["source"].isin(MCQ_SOURCES))


# Paires des QCM du train, comparées au corrigé
pd.crosstab(df_train["source"], gold_check(df_train), margins=True)

col_0,chosen fausse,les deux justes,lettre introuvable,vérifiée,All
source,,,,,
MedMCQA,304,12899,623,3728,17554
MedQA,134,8043,346,2280,10803
MedQA-Evol,68,7598,201,1372,9239
PubMedQA,30,2960,45,258,3293
TextBookQA,159,12801,332,2722,16014
All,695,44301,1547,10360,56903


Dans les QCM, la plupart des paires ont **deux réponses justes** : la préférence ne porte alors que sur la rédaction. Seules les paires **vérifiées** (`chosen` juste, `rejected` fausse) apprennent au modèle à préférer la bonne réponse à une erreur. Quelques centaines de paires ont même une `chosen` **fausse** : elles apprendraient une erreur au modèle.

In [11]:
# Longueur des textes (en caractères)
df_train[["prompt", "chosen", "rejected"]].apply(lambda col: col.str.len()).describe()

,prompt,chosen,rejected
count,109353.000000,109353.000000,109353.000000
mean,490.566852,2272.563771,1999.666575
std,341.991481,1216.098205,1291.313661
min,24.000000,15.000000,2.000000
25%,286.000000,1303.000000,931.000000
50%,402.000000,1937.000000,1570.000000
75%,627.000000,3318.000000,2995.000000
max,5509.000000,13054.000000,15940.000000


# Filtre sur label_type

- **train / validation** : on garde les types `LABEL_TYPES` (`hard` + `length`), et seulement les paires où la `chosen` a une note strictement supérieure à la `rejected`.
- **test** : gardé entier (tous les types, dont `human`). Ses étiquettes ont été revues par des experts : il sert de jeu d'évaluation clinique.

In [12]:
for name in ["train", "validation"]:
    df = splits[name]
    n_before = len(df)
    df = df[df["label_type"].isin(LABEL_TYPES) & (df["chosen_score"] > df["rejected_score"])]
    splits[name] = df
    print(f"{name}: {n_before} rows, {len(df)} après filtre sur label_type et sur la note")

train: 109353 rows, 80306 après filtre sur label_type et sur la note
validation: 2232 rows, 1639 après filtre sur label_type et sur la note


# Vérification par le corrigé

- **Sources QCM** : on ne garde que les paires vérifiées (`chosen` juste et `rejected` fausse d'après le corrigé).
- **Autres sources** (ChatDoctor, MedQuad, WikiInstruct…) : pas de corrigé. On ne garde que les paires où la `chosen` a une très bonne note (≥ `MIN_CHOSEN_SCORE`) et un écart net avec la `rejected` (≥ `MIN_SCORE_GAP`).

La colonne `pair_check` garde la trace du contrôle appliqué à chaque paire.

In [13]:
for name in ["train", "validation"]:
    df = splits[name]
    verified = gold_check(df) == "vérifiée"
    clear_preference = (
        ~df["source"].isin(MCQ_SOURCES)
        & (df["chosen_score"] >= MIN_CHOSEN_SCORE)
        & (df["chosen_score"] - df["rejected_score"] >= MIN_SCORE_GAP)
    )
    n_before = len(df)
    df = df.assign(pair_check=None)
    df.loc[verified, "pair_check"] = "golden_answer"
    df.loc[clear_preference, "pair_check"] = "gpt4_score"
    splits[name] = df.dropna(subset=["pair_check"])
    print(f"{name}: {n_before} rows, {len(splits[name])} après vérification")
    print(splits[name]["pair_check"].value_counts().to_string(), end="\n\n")

# Test : étiquettes revues par des experts
splits["test"] = splits["test"].assign(pair_check="expert_review")

train: 80306 rows, 10362 après vérification
pair_check
golden_answer    7243
gpt4_score       3119

validation: 1639 rows, 194 après vérification
pair_check
golden_answer    145
gpt4_score        49



# Nettoyage

In [14]:
for name, df in splits.items():
    n_before = len(df)
    df = df.assign(**{col: df[col].str.strip() for col in ["prompt", "chosen", "rejected"]})
    # Réponses vides ou identiques : pas de préférence à apprendre
    df = df[(df["chosen"] != "") & (df["rejected"] != "") & (df["chosen"] != df["rejected"])]
    # Une seule paire par question
    df = df.drop_duplicates(subset="prompt")
    splits[name] = df
    print(f"{name}: {n_before} rows, {len(df)} après nettoyage et dédoublonnage")

train: 10362 rows, 8879 après nettoyage et dédoublonnage
validation: 194 rows, 193 après nettoyage et dédoublonnage
test: 777 rows, 776 après nettoyage et dédoublonnage


In [15]:
# Fuite entre splits : une question vue à l'entraînement ne doit pas servir à l'évaluation
n_before = len(splits["train"])
held_out = pd.concat([splits["validation"]["prompt"], splits["test"]["prompt"]])
splits["train"] = splits["train"][~splits["train"]["prompt"].isin(held_out)]
splits["validation"] = splits["validation"][~splits["validation"]["prompt"].isin(splits["test"]["prompt"])]
print(f"train: {n_before - len(splits['train'])} questions présentes en validation / test supprimées")

train: 62 questions présentes en validation / test supprimées


# Filtre sur la longueur (tokens)

Les réponses sont longues (souvent plus de 2 000 caractères). Le DPO garde en mémoire les deux réponses de chaque paire : on limite la longueur pour tenir dans la mémoire du GPU (RTX 3080, 10 Go). Les longueurs sont comptées avec le tokenizer de Qwen3.

In [16]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def n_tokens(texts: pd.Series) -> list[int]:
    return [len(ids) for ids in tokenizer(texts.tolist(), add_special_tokens=False)["input_ids"]]


for name, df in splits.items():
    n_prompt, n_chosen, n_rejected = n_tokens(df["prompt"]), n_tokens(df["chosen"]), n_tokens(df["rejected"])
    df = df.assign(n_prompt=n_prompt, n_chosen=n_chosen, n_rejected=n_rejected)
    n_before = len(df)
    df = df[
        (df["n_prompt"] <= MAX_PROMPT_TOKENS)
        & (df["n_prompt"] + df[["n_chosen", "n_rejected"]].max(axis=1) <= MAX_LENGTH)
    ]
    splits[name] = df
    print(f"{name}: {n_before} rows, {len(df)} après filtre sur la longueur")

splits["train"][["n_prompt", "n_chosen", "n_rejected"]].describe().round(0)

train: 8817 rows, 8577 après filtre sur la longueur
validation: 193 rows, 191 après filtre sur la longueur
test: 776 rows, 756 après filtre sur la longueur


,n_prompt,n_chosen,n_rejected
count,8577.0,8577.0,8577.0
mean,117.0,329.0,374.0
std,77.0,216.0,198.0
min,6.0,30.0,7.0
25%,58.0,169.0,234.0
50%,102.0,251.0,312.0
75%,152.0,455.0,475.0
max,512.0,982.0,978.0


# Échantillonnage

Qualité plutôt que quantité : les filtres ont déjà écarté les paires douteuses. On plafonne le train à `N_TRAIN` paires et la validation à `N_VALIDATION` : quelques milliers de paires fiables suffisent pour le DPO d'un modèle de 1,7B. Le test est gardé entier.

In [17]:
splits["train"] = splits["train"].sample(n=min(N_TRAIN, len(splits["train"])), random_state=SEED)
splits["validation"] = splits["validation"].sample(n=min(N_VALIDATION, len(splits["validation"])), random_state=SEED)

for name, df in splits.items():
    print(f"{name}: {len(df)} rows, chosen plus longue dans {(df['n_chosen'] > df['n_rejected']).mean():.0%} des paires")

train: 2000 rows, chosen plus longue dans 26% des paires
validation: 191 rows, chosen plus longue dans 21% des paires
test: 756 rows, chosen plus longue dans 70% des paires


In [18]:
# Répartition finale par source et par type de paire
pd.concat({name: df["source"].value_counts() for name, df in splits.items()}, axis=1).fillna(0).astype(int)

,train,validation,test
source,,,
MedMCQA,512,46,74
TextBookQA,391,42,73
MedQA,306,25,74
WikiInstruct,250,16,79
MedQA-Evol,199,27,76
ChatDoctor,115,18,92
MedQuad,106,5,88
Medical-Instruct-120k,73,6,85
MedInstruct-52k,27,2,76


In [19]:
pd.concat({name: df["label_type"].value_counts() for name, df in splits.items()}, axis=1).fillna(0).astype(int)

,train,validation,test
label_type,,,
length,1852,177,173
hard,148,14,192
easy,0,0,234
human,0,0,157


In [20]:
pd.concat({name: df["pair_check"].value_counts() for name, df in splits.items()}, axis=1).fillna(0).astype(int)

,train,validation,test
pair_check,,,
golden_answer,1429,144,0
gpt4_score,571,47,0
expert_review,0,0,756


# Build DPO JSONL file from dataset

In [21]:
# Équivalent anglais du system prompt du SFT : même gabarit, langue des données
SYSTEM_PROMPT = "You are an expert medical assistant. Answer accurately and clearly."


def to_prompt(text: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]


def to_assistant(text: str) -> list[dict]:
    return [{"role": "assistant", "content": text}]


METADATA = [
    "prompt_id", "source", "label_type", "pair_check", "golden_answer",
    "chosen_model", "rejected_model", "chosen_score", "rejected_score",
]

# Un fichier JSONL par split : ../data/dpo_ultramedical_{train,validation,test}.jsonl
dpo_ds = DatasetDict()
for name, df in splits.items():
    df = df.assign(
        lang="en",
        prompt=df["prompt"].map(to_prompt),
        chosen=df["chosen"].map(to_assistant),
        rejected=df["rejected"].map(to_assistant),
    )
    dpo_ds[name] = Dataset.from_pandas(df[METADATA + ["lang", "prompt", "chosen", "rejected"]], preserve_index=False)
    dpo_ds[name].to_json(f"../data/dpo_ultramedical_{name}.jsonl", force_ascii=False)

dpo_ds["train"][0]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 16.71ba/s]


{'prompt_id': 'MedMCQA,109126',
 'source': 'MedMCQA',
 'label_type': 'length',
 'pair_check': 'golden_answer',
 'golden_answer': 'C',
 'chosen_model': 'Meta-Llama-3-70B-Instruct',
 'rejected_model': 'Meta-Llama-3-8B-Instruct',
 'chosen_score': 4.0,
 'rejected_score': 1.0,
 'lang': 'en',
 'prompt': [{'role': 'system',
   'content': 'You are an expert medical assistant. Answer accurately and clearly.'},
  {'role': 'user',
   'content': 'All of the following are characteristic of a bronchopulmonary segment, except:\n\nA. It is surgically respectableIt is surgically respectable\nB. It is named according to the segmental bronchus supplying it\nC. It is drained by intrasegmental branch of pulmonary vein.\nD. It is the largest subdivision of a lobe.'}],
 'chosen': [{'role': 'assistant',
   'content': "Let's break down each option:\n\nA. It is surgically respectable: This is true. Bronchopulmonary segments are the smallest lung units that can be removed surgically while maintaining the integri

In [22]:
print(dpo_ds)

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 191
    })
    test: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected'],
        num_rows: 756
    })
})


# Anonymisation des données

Seule la source **`ChatDoctor`** contient de vrais échanges patients / médecins, avec des noms (« Hello Shonda », « Dear Dr. … »). Les autres sources sont des questions d'examen (MedQA, MedMCQA, PubMedQA), des manuels, des articles Wikipédia et des réponses générées par des LLM : aucune donnée personnelle réelle.

On n'anonymise donc que `ChatDoctor` (paramètre `sources`), en anglais (`en_core_web_md`). Sur les autres sources, le modèle spaCy anglais ne produit presque que des faux positifs (gènes « Akt », « FOXO1 », éponymes « Wilms », options de QCM « A. Tuberous… ») et détruirait justement les réponses que le DPO compare.

In [23]:
# Sources contenant des échanges réels, donc des données personnelles possibles
ANONYMIZE_SOURCES = ["ChatDoctor"]

# Anonymisation de chaque split : ../data/anonymized/dpo_ultramedical_{split}.anonymized.jsonl
reports = {
    name: run(
        f"../data/dpo_ultramedical_{name}.jsonl", lang="en", output_dir="../data/anonymized", sources=ANONYMIZE_SOURCES
    )
    for name in dpo_ds
}
pd.DataFrame(reports).T[["records", "records_scanned", "records_modified", "persons_replaced"]]

,records,records_scanned,records_modified,persons_replaced
train,2000,115,72,208
validation,191,18,8,41
test,756,92,56,141


In [24]:
# DataFrame des champs "content" des jsonl anonymisés : une ligne par message (hors system)
rows = []
for name in dpo_ds:
    with open(f"../data/anonymized/dpo_ultramedical_{name}.anonymized.jsonl", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            for field in ["prompt", "chosen", "rejected"]:
                for message in record[field]:
                    if message["role"] != "system":
                        rows.append(
                            {"split": name, "source": record["source"], "field": field, "content": message["content"]}
                        )

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
pd.crosstab(contents_df["source"], contents_df["field"], values=contents_df["n_person"], aggfunc="sum")

8841 messages, dont 217 avec au moins une balise <PERSON>


field,chosen,prompt,rejected
source,,,
ChatDoctor,179,49,158
MedInstruct-52k,0,0,0
MedMCQA,0,0,0
MedQA,0,0,0
MedQA-Evol,0,0,0
MedQuad,0,0,0
Medical-Instruct-120k,0,0,0
PubMedQA,0,0,0
TextBookQA,0,0,0


In [25]:
def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["split", "source", "field"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(min(20, len(occurrences_df)), random_state=SEED)

386 occurrences de <PERSON>


,split,source,field,context
6725,test,ChatDoctor,rejected,"g clomiphene citrate and contact your healthcare provider. <PERSON> is generally considered safe for use in pregnant women, and"
6671,test,ChatDoctor,rejected,"ke deep breathing, meditation, or reading. 5. **Monitor and <PERSON>: \t* Keep a symptom diary: Record her symptoms, medication u"
3514,train,ChatDoctor,chosen,"poultry, fish, beans, lentils, and fortified cereals. 2. **<PERSON> foods**: Include vitamin C-rich foods like citrus fruits, b"
6598,test,ChatDoctor,chosen,n provide you with some general information on this topic. <PERSON> is a condition where the veins within the scrotum become en
5170,train,ChatDoctor,chosen,"fish, eggs, dairy products, and legumes in your diet. 3. **<PERSON>* Vitamin C is important for collagen synthesis and wound he"
6659,test,ChatDoctor,rejected,"al lobe, indicating an old intra-cerebral hemorrhage. Smt. <PERSON> has a history of seizure disorder, hypertension, and aspira"
3106,train,ChatDoctor,chosen,", let's break down what each of these values means: * ALT (<PERSON>): 63 is slightly elevated, but within the normal range. Thi"
3961,train,ChatDoctor,chosen,"you're taking thyroxine (T4) to manage your hypothyroidism. <PERSON>, your TSH level of 7.7 indicates that your thyroid hormone"
6677,test,ChatDoctor,rejected,Here's a brief explanation of each: 1. **Todd's Paralysis (<PERSON>)**: This is a transient condition often associated with sei
1091,train,ChatDoctor,rejected,"ying cause of scabies but can help manage the symptoms. 3. <PERSON>: I'm not familiar with a medication called Droxaryl, and it"


# Contrôle final

In [26]:
# Rechargement des fichiers anonymisés, tels que les lira le notebook d'entraînement DPO
DATA_FILES = {name: f"../data/anonymized/dpo_ultramedical_{name}.anonymized.jsonl" for name in dpo_ds}
final_ds = load_dataset("json", data_files=DATA_FILES)
print(final_ds)

for name, split in final_ds.items():
    same = sum(c[0]["content"] == r[0]["content"] for c, r in zip(split["chosen"], split["rejected"]))
    assert same == 0, f"{name}: {same} paires avec chosen == rejected"
print("OK : aucune paire avec chosen == rejected")

Generating train split: 2000 examples [00:00, 40749.49 examples/s]
Generating validation split: 191 examples [00:00, 17719.01 examples/s]
Generating test split: 756 examples [00:00, 33934.35 examples/s]

DatasetDict({
    train: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 191
    })
    test: Dataset({
        features: ['prompt_id', 'source', 'label_type', 'pair_check', 'golden_answer', 'chosen_model', 'rejected_model', 'chosen_score', 'rejected_score', 'lang', 'prompt', 'chosen', 'rejected', 'persons_replaced'],
        num_rows: 756
    })
})
OK : aucune paire avec chosen == rejected


# Limites

- **Préférences** : vérifiées par le corrigé pour les QCM (`pair_check = golden_answer`), jugées par GPT-4 seul pour les autres sources (`gpt4_score`, biais possible en faveur des réponses de GPT-4). Seul le test a été revu par des experts.
- **Anglais uniquement** : couvre la partie anglophone du corpus bilingue. La partie française du DPO reste à constituer.
- **Écart de style avec le SFT** : réponses longues et rédigées, alors que le SFT MediQAl produit des réponses courtes. Ajouter de l'anglais au SFT (MedQuAD) avant le DPO limite cet écart.
- **Anonymisation limitée à `ChatDoctor`** : choix à justifier dans la documentation RGPD (absence de données personnelles réelles dans les autres sources). Même sur `ChatDoctor`, des noms de médicaments (« Winflam », « Pyzina ») sont pris pour des personnes : relire l'échantillon ci-dessus et compléter `EXCLUDED_WORDS` dans `rgpd.py`.